In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely import wkb, wkt
from libpysal.weights import DistanceBand
#import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
from libpysal.weights import DistanceBand, lag_spatial
from sklearn.preprocessing import StandardScaler

In [5]:
import statsmodels.api as sm

In [6]:
g  = gpd.read_file("../Data/Shapefiles/APES_metric.gpkg", layer="nuts3")
# Replico i filtri: geometrie non vuote e RecordType != "ZIKV"
g = g.loc[(~g.geometry.is_empty) & (g["RecordType"] != "ZIKV")].copy()

# ------------------------------------------------------------------
# 3) PREPARA DATASET come in R (selezione variabili e aggregazione)
# ------------------------------------------------------------------
avg_var = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
           "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
           "Mean_T_Spring", "Max_DD_Spring",
           "Mean_T_Summer",
           "Max_WD_Autumn", "Max_DD_Autumn",
           "gdp", "pop_dens"]

cols_keep = (["incidence"] + avg_var +
             ["geometry", "NUTS_NAME", "NUTS_CODE"])

df = g[cols_keep].copy()

# In R raggruppi per geometry, NUTS_NAME, NUTS_CODE; in GeoPandas
# conviene usare l'identificativo NUTS_CODE (che è unico) e conservare la prima geometria.
agg_dict = {"incidence": "sum"}
agg_dict.update({v: "mean" for v in avg_var})
agg_dict.update({"geometry": "first", "NUTS_NAME": "first"})

fin = (
    df.groupby("NUTS_CODE", as_index=False)
      .agg(agg_dict)
      .set_geometry("geometry")
)
fin = gpd.GeoDataFrame(fin, geometry="geometry", crs=g.crs)

# Centroidi per controllo/coords (come in R)
fin["cent"] = fin.geometry.centroid

# ------------------------------------------------------------------
# 4) STANDARDIZZA (z-score) tutte le variabili numeriche
# ------------------------------------------------------------------
num_cols = fin.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
fin_scaled = fin.copy()
fin_scaled[num_cols] = scaler.fit_transform(fin_scaled[num_cols])

# ------------------------------------------------------------------
# 5) RIPROIEZIONE a metrica (EPSG:32633 ~ UTM 33N), come in R
# ------------------------------------------------------------------
fin_m = fin_scaled.to_crs(32633)

# ------------------------------------------------------------------
# 6) COSTRUISCI matrici di vicinato entro 150 km e Y laggata
# ------------------------------------------------------------------
# In R, coordinates() sullo Spatial di poligoni restituisce i centroidi.
# Qui uso i centroidi in metri (proiezione UTM 33N).
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

# threshold 150 km, binaria, senza diagonale
W = DistanceBand(coords, threshold=150000, binary=True, silence_warnings=True)
# trasformo in matrice sparsa CSR (se serve), o applico direttamente come W * y
# libpysal conserva W in forma weights + neighbors; uso full() per matrice densa se serve
W_sparse = W.sparse  # scipy CSR matrix

# y laggata
y = fin_m["incidence"].to_numpy().reshape(-1, 1)
y_lag = W_sparse.dot(y).ravel()
fin_m["Y_weighted"] = y_lag

# ------------------------------------------------------------------
# 7) STIMA OLS con formula analoga all’R (Mean_T_Summer esclusa come nel codice R)
# ------------------------------------------------------------------
# NOTA: qui stai regredendo Y_lag su X (non è una SAR/SLX); è OLS classico.
formula = ("Y_weighted ~ Forest + Shrub + Urban + Crop + Water + Other + "
           "Mean_P_Winter + Max_WD_Winter + Max_DD_Winter + "
           "Mean_T_Spring + Max_DD_Spring + "
           # "Mean_T_Summer + "  # esclusa come in R
           "Max_WD_Autumn + Max_DD_Autumn + gdp + pop_dens")

# statsmodels richiede un DataFrame "piatto" (niente geometry)
Xy = pd.DataFrame(fin_m.drop(columns=["geometry", "cent"]))
lm_fit = smf.ols(formula, data=Xy).fit()

# ------------------------------------------------------------------
# 8) VIF per multicollinearità globale (come car::vif)
# ------------------------------------------------------------------
# Creo la matrice di design senza l'intercetta per VIF
design_cols = ["Forest", "Shrub", "Urban", "Crop", "Water", "Other",
               "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
               "Mean_T_Spring", "Max_DD_Spring",
               "Max_WD_Autumn", "Max_DD_Autumn",
               "gdp", "pop_dens"]
X_for_vif = Xy[design_cols].copy()
X_for_vif = sm.add_constant(X_for_vif, has_constant="add")  # per sicurezza



# ------------------------------------------------------------------
# 9) Tabella coefficienti formattata come in R
# ------------------------------------------------------------------
coefs = lm_fit.summary2().tables[1].reset_index().rename(columns={"index": "term"})
coefs["estimate"] = coefs["Coef."].round(2)
coefs["std.error"] = coefs["Std.Err."].round(2)

def fmt_p(p):
    try:
        return "<.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return str(p)

coefs["p-value"] = coefs["P>|t|"].apply(fmt_p)

lm_table = coefs.loc[:, ["term", "estimate", "std.error", "p-value"]]
lm_table = lm_table.rename(columns={
    "term": "Covariate",
    "estimate": "Estimate",
    "std.error": "SE"
})

print("\n=== Coefficient table ===")
print(lm_table.to_string(index=False))
# Se vuoi salvare:

Q:\UserTemp\massaro\AppData\Local\Temp\272\ipykernel_73076\2793943224.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fin["cent"] = fin.geometry.centroid



=== Coefficient table ===
    Covariate  Estimate   SE p-value
    Intercept     -4.12 0.25   <.001
       Forest      0.19 0.34   0.581
        Shrub      1.52 0.29   <.001
        Urban      0.76 0.27   0.004
         Crop      0.57 0.33   0.091
        Water      0.76 0.27   0.005
        Other      0.54 0.30   0.069
Mean_P_Winter      3.94 0.39   <.001
Max_WD_Winter     -2.42 0.52   <.001
Max_DD_Winter      0.09 0.41   0.819
Mean_T_Spring      2.94 0.45   <.001
Max_DD_Spring     -2.98 0.37   <.001
Max_WD_Autumn     -1.43 0.43   <.001
Max_DD_Autumn      0.94 0.45   0.036
          gdp     -2.48 0.32   <.001
     pop_dens      1.04 0.27   <.001


In [7]:
# Extract variables
X = fin_m[design_cols].to_numpy()
y = fin_m["incidence"].to_numpy().reshape(-1, 1)
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])


In [ ]:
np.savez("../Data/processed_data_for_gwr.npz", X=X, y=y, coords=coords, columns=design_cols)


In [8]:
import os

In [9]:
# p-value formatter (same style as your table)
def fmt_p(p):
    try:
        return "<.001" if p < 0.001 else f"{p:.3f}"
    except Exception:
        return str(p)

# thresholds (meters): 100km .. 200km by 10km
thresholds = np.arange(100_000, 200_000 + 1, 10_000)

# response is fixed across thresholds
y = fin_m["incidence"].to_numpy().reshape(-1, 1)

all_tables = []  # collect all coefficient tables here

for thr in thresholds:
    # 1) Build W at this threshold
    W = DistanceBand(coords, threshold=thr, binary=True, silence_warnings=True)

    # 2) Spatial lag of y (weights * y)
    y_lag = W.sparse.dot(y).ravel()

    # 3) Create flat DataFrame for statsmodels
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 4) Fit OLS with your same formula
    lm_fit_thr = smf.ols(formula, data=Xy_thr).fit()

    # 5) Build coefficient table like your R-style formatting
    coefs = lm_fit_thr.summary2().tables[1].reset_index().rename(columns={"index": "term"})
    coefs["estimate"] = coefs["Coef."].round(2)
    coefs["std.error"] = coefs["Std.Err."].round(2)
    coefs["p-value"] = coefs["P>|t|"].apply(fmt_p)

    lm_table = coefs.loc[:, ["term", "estimate", "std.error", "p-value"]]
    lm_table = lm_table.rename(columns={"term": "Covariate", "estimate": "Estimate", "std.error": "SE"})

    # 6) Print nicely
    print(f"\n=== Distance threshold = {thr/1000:.0f} km ===")
    print(lm_table.to_string(index=False))

    # 7) Save per-threshold CSV
    lm_table_out = lm_table.copy()
    lm_table_out.insert(0, "threshold_m", thr)





=== Distance threshold = 100 km ===
    Covariate  Estimate   SE p-value
    Intercept     -1.91 0.14   <.001
       Forest     -0.04 0.20   0.820
        Shrub      0.76 0.17   <.001
        Urban      0.35 0.15   0.022
         Crop      0.33 0.19   0.087
        Water      0.39 0.15   0.011
        Other      0.37 0.17   0.030
Mean_P_Winter      1.72 0.23   <.001
Max_WD_Winter     -1.01 0.30   <.001
Max_DD_Winter      0.22 0.23   0.339
Mean_T_Spring      1.74 0.26   <.001
Max_DD_Spring     -1.62 0.21   <.001
Max_WD_Autumn     -0.71 0.25   0.004
Max_DD_Autumn      0.25 0.26   0.334
          gdp     -1.18 0.18   <.001
     pop_dens      0.51 0.16   0.001

=== Distance threshold = 110 km ===
    Covariate  Estimate   SE p-value
    Intercept     -2.33 0.16   <.001
       Forest     -0.04 0.22   0.845
        Shrub      0.88 0.18   <.001
        Urban      0.41 0.17   0.015
         Crop      0.37 0.21   0.085
        Water      0.46 0.17   0.007
        Other      0.40 0.19   0.034
M

In [10]:
from libpysal.weights import DistanceBand, KNN, lag_spatial

In [11]:
y_vec =fin_m["incidence"].to_numpy().reshape(-1, 1)

for thr in thresholds:
    # 1) Build W at this threshold (row-standardize so rows sum to 1)
    W = DistanceBand(coords, threshold=thr, binary=True, silence_warnings=True)
    W.transform = "R"  # row-standardize

    # 2) Spatial lag of y as neighbor AVERAGE (thanks to transform='R')
    y_lag = lag_spatial(W, y_vec)
    #y_lag = W.sparse.dot(y).ravel()

    # 3) z-score the spatial lag so the intercept ~ 0 and effects are SD(Y) per SD(X)
    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std

    # 4) Create flat DataFrame for statsmodels (same as your OLS)
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 5) Fit OLS with your same formula
    lm_fit_thr = smf.ols(formula, data=Xy_thr).fit()

    # 6) Coefficient table (same formatting)
    coefs = lm_fit_thr.summary2().tables[1].reset_index().rename(columns={"index": "term"})
    coefs["estimate"]   = coefs["Coef."].round(2)
    coefs["std.error"]  = coefs["Std.Err."].round(2)
    coefs["p-value"]    = coefs["P>|t|"].apply(fmt_p)

    lm_table = coefs.loc[:, ["term", "estimate", "std.error", "p-value"]]
    lm_table = lm_table.rename(columns={"term": "Covariate", "estimate": "Estimate", "std.error": "SE"})

    # 7) Print nicely
    print(f"\n=== Distance threshold = {thr/1000:.0f} km (row-std W; z-scored Wy) ===")
    print(lm_table.to_string(index=False))



=== Distance threshold = 100 km (row-std W; z-scored Wy) ===
    Covariate  Estimate   SE p-value
    Intercept      0.01 0.03   0.751
       Forest     -0.10 0.04   0.007
        Shrub      0.24 0.03   <.001
        Urban     -0.00 0.03   0.986
         Crop     -0.08 0.04   0.025
        Water      0.11 0.03   <.001
        Other      0.01 0.03   0.652
Mean_P_Winter      0.10 0.04   0.022
Max_WD_Winter     -0.07 0.06   0.197
Max_DD_Winter     -0.04 0.04   0.344
Mean_T_Spring      0.35 0.05   <.001
Max_DD_Spring     -0.31 0.04   <.001
Max_WD_Autumn     -0.02 0.05   0.614
Max_DD_Autumn      0.06 0.05   0.204
          gdp     -0.11 0.04   0.001
     pop_dens      0.05 0.03   0.114

=== Distance threshold = 110 km (row-std W; z-scored Wy) ===
    Covariate  Estimate   SE p-value
    Intercept      0.01 0.03   0.735
       Forest     -0.10 0.04   0.005
        Shrub      0.23 0.03   <.001
        Urban      0.00 0.03   0.922
         Crop     -0.09 0.04   0.017
        Water      0.10 0

In [12]:
import statsmodels.formula.api as smf
from libpysal.weights import DistanceBand
from esda.moran import Moran
import matplotlib.pyplot as plt

In [13]:
# columns needed by the model (no intercept) + response placeholder
design_cols = [
    "Forest", "Shrub", "Urban", "Crop", "Water", "Other",
    "Mean_P_Winter", "Max_WD_Winter", "Max_DD_Winter",
    "Mean_T_Spring", "Max_DD_Spring",
    "Max_WD_Autumn", "Max_DD_Autumn",
    "gdp", "pop_dens"
]

metrics = []
coords = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

for thr in thresholds:
    # 1) Build W on ALL rows first
    W_full = DistanceBand(coords, threshold=thr, binary=True, silence_warnings=True)
    # row-standardize for Moran
    W_full.transform = "R"

    # 2) Compute Y lag on ALL rows
    y_full = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag_full = W_full.sparse.dot(y_full).ravel()

    # 3) z-score the spatial lag so the intercept ~ 0 and effects are SD(Y) per SD(X)
    y_mean = y_lag_full.mean()
    y_std  = y_lag_full.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std

    
    # 3) Assemble a temp DF with ALL rows
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z

    # 4) Build a CLEAN mask: only rows with complete data for model
    needed = ["Y_weighted"] + design_cols
    mask = tmp[needed].notna().all(axis=1).to_numpy()

    # If nothing to fit, skip
    if mask.sum() == 0:
        print(f"[WARN] {thr/1000:.0f} km → no complete cases. Skipping.")
        continue

    # 5) Subset DF and COORDS and W to the SAME rows
    tmp_clean = tmp.loc[mask].copy()
    coords_clean = coords[mask]

    # Rebuild W on the subset (simplest & safest)
    W_sub = DistanceBand(coords_clean, threshold=thr, binary=True, silence_warnings=True)
    W_sub.transform = "R"

    # Diagnostics about network structure
    n_islands = len(getattr(W_sub, "islands", []))
    mean_neighbors = float(np.mean([len(nbrs) for nbrs in W_sub.neighbors.values()]))

    # 6) Flat DF (drop geometry) and fit OLS
    Xy_thr = pd.DataFrame(tmp_clean.drop(columns=[c for c in ["geometry", "cent"] if c in tmp_clean.columns]))
    lm_fit_thr = smf.ols(formula, data=Xy_thr).fit()

    # 7) Residual Moran’s I (dimensions now match)
    resid = lm_fit_thr.resid.values  # length == W_sub.n
    mi = Moran(resid, W_sub)  # permutations=999 by default

    metrics.append({
        "threshold_m": thr,
        "threshold_km": thr / 1000,
        "r2": lm_fit_thr.rsquared,
        "adj_r2": lm_fit_thr.rsquared_adj,
        "aic": lm_fit_thr.aic,
        "bic": lm_fit_thr.bic,
        "moran_I": mi.I,
        "moran_p": mi.p_sim,
        "n_obs": int(mask.sum()),
        "n_islands": n_islands,
        "mean_neighbors": mean_neighbors
    })

# ----------------------------
# Save metrics and quick plots
# ----------------------------
m = pd.DataFrame(metrics)




In [ ]:
# Simple plots
def _plot(x, y, ylabel, title, fname):
    out_dir = '../Figures/'
    plt.subplots()
    plt.plot(m[x], m[y], marker="o")
    plt.xlabel("Distance threshold (km)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, fname), dpi=150)
    plt.show()

In [ ]:

_plot("threshold_km", "r2", "R²", "R² vs distance threshold", "r2_vs_threshold.png")


In [ ]:
_plot("threshold_km", "adj_r2", "Adjusted R²", "Adjusted R² vs distance threshold", "adjr2_vs_threshold.png")

In [ ]:
_plot("threshold_km", "aic", "AIC (lower is better)", "AIC vs distance threshold", "aic_vs_threshold.png")


In [ ]:
_plot("threshold_km", "moran_I", "Moran's I (residuals)", "Residual Moran's I vs distance threshold", "moranI_vs_threshold.png")


In [ ]:
_plot("threshold_km", "mean_neighbors", "Mean # neighbors", "Mean neighbors vs distance threshold", "mean_neighbors_vs_threshold.png")

In [ ]:
if not m.empty:
    best_aic_row = m.loc[m["aic"].idxmin()]
    best_moran_row = m.iloc[(m["moran_I"].abs()).idxmin()]
    print("\n=== Summary ===")
    print("Best (min) AIC at ~{:.0f} km: AIC={:.2f}, R²={:.3f}, adj R²={:.3f}, Moran's I={:.3f} (p={:.3f})"
          .format(best_aic_row["threshold_km"], best_aic_row["aic"], best_aic_row["r2"],
                  best_aic_row["adj_r2"], best_aic_row["moran_I"], best_aic_row["moran_p"]))
    print("Smallest |Moran's I| at ~{:.0f} km: I={:.3f} (p={:.3f}), AIC={:.2f}, adj R²={:.3f}, n_obs={}"
          .format(best_moran_row["threshold_km"], best_moran_row["moran_I"],
                  best_moran_row["moran_p"], best_moran_row["aic"], best_moran_row["adj_r2"],
                  int(best_moran_row["n_obs"])))

In [14]:
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

In [17]:
kernels = ["gaussian", "bisquare", "exponential"]   # add/remove kernels as you like

In [18]:
coords_all = np.column_stack([fin_m.geometry.centroid.x, fin_m.geometry.centroid.y])

def get_aicc_safe(fit):
    # Try common names across mgwr versions
    for name in ("aicc", "AICc", "aic_c"):
        if hasattr(fit, name):
            try:
                return float(getattr(fit, name))
            except Exception:
                pass
    # Fallback to AIC if nothing else exists
    if hasattr(fit, "aic"):
        try:
            return float(getattr(fit, "aic"))
        except Exception:
            pass
    return np.nan

rows = []
best = None

for d in thresholds:
    # 1) Build Y_weighted EXACTLY like your OLS at this distance
    Wd = DistanceBand(coords_all, threshold=d, binary=True, silence_warnings=True)
    y_raw = fin_m["incidence"].to_numpy().reshape(-1, 1)
    y_lag = Wd.sparse.dot(y_raw).ravel()

    y_mean = y_lag.mean()
    y_std  = y_lag.std(ddof=0)
    if y_std == 0:
        print(f"[WARN] {thr/1000:.0f} km → zero variance in Wy; skipping.")
        continue
    y_lag_z = (y_lag - y_mean) / y_std
    
    tmp = fin_m.copy()
    tmp["Y_weighted"] = y_lag_z
    Xy_thr = pd.DataFrame(tmp.drop(columns=[c for c in ["geometry", "cent"] if c in tmp.columns]))

    # 2) Complete-case mask and align arrays
    need_cols = ["Y_weighted"] + design_cols
    mask = Xy_thr[need_cols].notna().all(axis=1).to_numpy()
    if mask.sum() == 0:
        print(f"[WARN] {d/1000:.0f} km: no complete cases. Skipping.")
        continue

    coords = coords_all[mask]
    y = Xy_thr.loc[mask, "Y_weighted"].to_numpy().reshape(-1, 1)
    X = Xy_thr.loc[mask, design_cols].to_numpy()
    n = X.shape[0]

    row_sums = np.asarray(Wd.sparse.sum(axis=1)).ravel()
    isolated = int((row_sums[mask] == 0).sum())

    for ker in kernels:
        try:
            # Bandwidth search (adaptive neighbor count)
            bw = Sel_BW(coords, y, X, kernel=ker, fixed=False, spherical=False).search()

            fit = GWR(coords, y, X, bw,
                      kernel=ker, fixed=False, constant=True, spherical=False).fit()

            aicc_val = get_aicc_safe(fit)
            mean_r2 = float(np.mean(fit.localR2)) if hasattr(fit, "localR2") else np.nan
            min_r2  = float(np.min(fit.localR2))  if hasattr(fit, "localR2") else np.nan
            max_r2  = float(np.max(fit.localR2))  if hasattr(fit, "localR2") else np.nan

            row = {
                "kernel": ker,
                "fixed": False,
                "distance_m_for_Y": d,
                "bandwidth_neighbors": int(bw) if bw is not None else np.nan,
                "AICc": aicc_val,
                "mean_localR2": mean_r2,
                "min_localR2":  min_r2,
                "max_localR2":  max_r2,
                "n": n,
                "isolated_for_Y": isolated
            }
            rows.append(row)

            # Track best by AICc if finite
            if np.isfinite(row["AICc"]) and (best is None or row["AICc"] < best["meta"]["AICc"]):
                best = {"meta": row, "fit": fit, "mask": mask}

            print(f"d={d/1000:.0f} km | {ker:10s} | bw={bw} | "
                  f"AICc={aicc_val if np.isfinite(aicc_val) else 'NA'} | "
                  f"R2 μ/lo/hi={mean_r2:.3f}/{min_r2:.3f}/{max_r2:.3f} | "
                  f"n={n} isoY={isolated}")

        except Exception as e:
            rows.append({
                "kernel": ker, "fixed": False, "distance_m_for_Y": d,
                "bandwidth_neighbors": np.nan, "AICc": np.nan,
                "mean_localR2": np.nan, "min_localR2": np.nan, "max_localR2": np.nan,
                "n": n, "isolated_for_Y": isolated, "error": str(e)
            })
            print(f"d={d/1000:.0f} km | {ker:10s} | ERROR: {e}")

# Save summary
res_df = pd.DataFrame(rows)

d=100 km | gaussian   | bw=73.0 | AICc=2041.3346344392476 | R2 μ/lo/hi=0.742/0.442/0.867 | n=1134 isoY=27
d=100 km | bisquare   | bw=100.0 | AICc=1888.9123553840982 | R2 μ/lo/hi=0.764/-1.713/0.963 | n=1134 isoY=27
d=100 km | exponential | bw=73.0 | AICc=2059.631755189019 | R2 μ/lo/hi=0.751/0.492/0.858 | n=1134 isoY=27
d=110 km | gaussian   | bw=73.0 | AICc=1919.4756376622909 | R2 μ/lo/hi=0.757/0.459/0.882 | n=1134 isoY=11
d=110 km | bisquare   | bw=100.0 | AICc=1740.2216504991084 | R2 μ/lo/hi=0.779/-0.885/0.971 | n=1134 isoY=11
d=110 km | exponential | bw=73.0 | AICc=1953.8131117450594 | R2 μ/lo/hi=0.765/0.508/0.874 | n=1134 isoY=11
d=120 km | gaussian   | bw=73.0 | AICc=1866.4211120109976 | R2 μ/lo/hi=0.765/0.470/0.888 | n=1134 isoY=7
d=120 km | bisquare   | bw=100.0 | AICc=1673.819976154171 | R2 μ/lo/hi=0.781/-1.070/0.967 | n=1134 isoY=7
d=120 km | exponential | bw=73.0 | AICc=1888.9544822387998 | R2 μ/lo/hi=0.774/0.530/0.879 | n=1134 isoY=7
d=130 km | gaussian   | bw=73.0 | AICc=181

In [19]:
Xy_thr

,NUTS_CODE,incidence,Forest,Shrub,Urban,Crop,Water,Other,Mean_P_Winter,Max_WD_Winter,Max_DD_Winter,Mean_T_Spring,Max_DD_Spring,Mean_T_Summer,Max_WD_Autumn,Max_DD_Autumn,gdp,pop_dens,NUTS_NAME,Y_weighted
0,AL011,-0.179151,0.756189,0.509149,-0.026845,0.043578,0.086687,0.775878,3.112144,1.945627,-0.714194,-0.550112,-1.092503,-0.157436,0.758760,0.285802,-1.400183,-0.341147,Dibër,1.084448
1,AL012,-0.179151,-0.619313,-0.208217,-0.565779,-0.567606,-0.268374,-0.608400,3.359616,1.647878,-0.206647,1.174900,0.627053,1.258868,0.349179,0.488145,-1.341374,-0.025330,Durrës,0.248956
2,AL014,-0.179151,-0.093136,-0.029470,-0.644679,-0.524828,-0.114005,-0.561701,3.259787,1.399755,-0.511175,0.473871,-0.101889,0.709290,0.553970,0.488145,-1.401131,-0.314489,Lezhë,0.248956
3,AL015,-0.091608,0.511414,0.342044,-0.021393,-0.275947,1.519490,-0.158081,8.237140,2.636556,-0.735017,0.912847,-0.712936,0.838336,0.262537,0.425886,-1.396199,-0.333573,Shkodër,0.222926
4,AL021,-0.179151,1.286983,0.588713,-0.098131,0.813643,-0.189353,0.247437,1.844104,0.804258,-0.443502,-0.077223,-0.457015,0.341495,-0.060402,0.892831,-1.407771,-0.306361,Elbasan,1.960832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1169,SK023,-0.179151,-0.050808,-0.266253,1.778052,2.121404,0.081201,-0.584426,-0.808190,-1.428856,0.131717,0.690843,0.103710,0.384495,-1.135554,0.806112,-0.825375,-0.290502,Nitriansky kraj,0.709045
1170,SK031,-0.179151,3.125244,-0.242245,0.942352,-0.219131,0.270080,0.861353,0.031946,1.449379,-1.458596,-1.194963,-1.223339,-1.205290,0.912353,0.054553,-0.795971,-0.296198,Žilinský kraj,0.183374
1171,SK032,-0.179151,3.720250,-0.234531,0.928419,1.080813,-0.242623,0.086830,-0.552274,-0.237862,-0.206647,-0.291907,-0.606542,-0.502389,-0.623577,0.603770,-0.931610,-0.327001,Banskobystrický kraj,0.560546
1172,SK041,-0.179151,3.406806,-0.216848,1.656421,0.890459,-0.165417,0.628226,-0.729122,-0.684485,-1.154068,-0.650596,-1.185957,-0.763486,-0.162798,0.112365,-1.028360,-0.305561,Prešovský kraj,0.450968
